# 4.0 — Baselines clasificación multiclase (nonMalignant vs tipos de cáncer)

Este notebook replica el flujo de la iteración binaria, pero con etiquetas multiclase:
- `nonMalignant` (todas las patologías no-cáncer agrupadas)
- `Patient_group` para muestras `Malignant` (tipo de cáncer)

Además, se usa un **umbral sobre p(cáncer)** (`1 - p(nonMalignant)`) para controlar falsos negativos.

In [ ]:
from __future__ import annotations
from pathlib import Path
from dataclasses import replace
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from genomics_dl.models.train_multiclass import MulticlassTrainConfig, run_training

warnings.filterwarnings("ignore", category=ConvergenceWarning)

logging.getLogger("alembic").setLevel(logging.ERROR)
logging.getLogger("alembic.runtime.migration").setLevel(logging.ERROR)

logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)


### Rutas y carga de datos

In [3]:
# Paths
DATA_PROCESSED = Path("../data/processed")
TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train.parquet"
TEST_PATH  = DATA_PROCESSED / "gse183635_tep_tpm_test.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

df_train.shape, df_test.shape

((1880, 5452), (471, 5452))

### Separación genes vs metadatos

In [4]:
metadata_cols = [
    "Sample ID",
    "Patient_group",
    "Stage",
    "Sex",
    "Age",
    "Sample-supplying institution",
    "Training series",
    "Evaluation series",
    "Validation series",
    "lib.size",
    "classificationScoreCancer",
    "Class_group",
]

# Genes: columnas ENSG...
gene_cols = [c for c in df_train.columns if str(c).startswith("ENSG")]

assert "Class_group" in df_train.columns
assert "Patient_group" in df_train.columns
assert len(gene_cols) > 0
assert set(gene_cols).isdisjoint(set(metadata_cols))

len(gene_cols), gene_cols[:5]

(5440,
 ['ENSG00000000419',
  'ENSG00000000460',
  'ENSG00000000938',
  'ENSG00000001036',
  'ENSG00000001461'])

### Sanity check de etiquetas (train)

In [5]:
df_train["Class_group"].value_counts(dropna=False)

Class_group
Malignant       1302
nonMalignant     578
Name: count, dtype: int64

In [ ]:
df_train.loc[df_train["Class_group"].astype(str) == "Malignant", "Patient_group"].value_counts().head(20)

Patient_group
Non-small-cell lung cancer    417
Ovarian cancer                114
Glioma                        113
Pancreatic cancer              93
Breast cancer                  80
Head and neck cancer           79
Cholangiocarcinoma             71
Colorectal cancer              69
Melanoma                       54
Sarcoma                        44
Endometrial cancer             34
Prostate cancer                23
Multiple Myeloma               22
Urothelial cancer              22
Renal cell cancer              20
Hepatocellular carcinoma       19
Lymphoma                       16
Esophageal carcinoma           12
Name: count, dtype: int64

## Experimentos baseline (sweep)

Ranking:
1) Minimizar `test_cancer_fn` (cáncer predicho como nonMalignant)
2) Maximizar `test_cancer_recall_sensitivity`
3) Maximizar `test_f1_macro`

In [ ]:
def slugify_token(value):
    return str(value).replace(".", "p").replace("-", "m")

def build_model_name(clf_name, feat_cfg, malignant_weight, variant_tag):
    return "_".join([
        clf_name,
        f"pca{int(feat_cfg['use_pca'])}",
        f"log{int(feat_cfg['selector_on_log'])}",
        f"vq{int(feat_cfg['var_quantile']*100)}",
        f"mw{slugify_token(malignant_weight)}",
        variant_tag,
    ])

# Base config (ligero)
base_cfg = MulticlassTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    model_name="multiclass_smoke",
    model_version="v0.1.0",
    use_pca=False,
    var_quantile=0.2,
    selector_on_log=False,
    pca_var_threshold=0.9,
    cv_splits=3,
    min_cancer_recall_for_threshold=0.95,
    threshold_objective="specificity",
    experiment_name="gse183635_multiclass_smoke",
    save_local_bundle=False,
    save_plots=False,
)

# Sweep (muy pequeño)
feat_grid = [
    dict(use_pca=False, selector_on_log=False, var_quantile=0.2, pca_var_threshold=0.9),
]

clf_grid = [
    ("logreg", dict(solver="lbfgs", max_iter=2000)),
    ("linear_svc_calibrated", dict(C=1.0)),
]

malignant_weights = [1.0, 2.0]

sweep = []
for feat_cfg in feat_grid:
    for clf_name, clf_params in clf_grid:
        for mw in malignant_weights:
            sweep.append(dict(feat_cfg=feat_cfg, clf_name=clf_name, clf_params=clf_params, mw=mw))

len(sweep)

4

In [8]:
results = []
for combo in sweep:
    feat_cfg = combo["feat_cfg"]
    clf_name = combo["clf_name"]
    clf_params = combo["clf_params"]
    mw = combo["mw"]

    model_name = build_model_name(clf_name, feat_cfg, mw, variant_tag="sweep")
    cfg = replace(
        base_cfg,
        model_name=model_name,
        clf_name=clf_name,
        clf_params=clf_params,
        malignant_weight=mw,
        use_pca=feat_cfg["use_pca"],
        selector_on_log=feat_cfg["selector_on_log"],
        var_quantile=feat_cfg["var_quantile"],
        pca_var_threshold=feat_cfg["pca_var_threshold"],
    )

    out = run_training(cfg, feature_cols=gene_cols)
    tm = out["test_metrics"]
    results.append({
        "model_name": model_name,
        "clf_name": clf_name,
        "mw": mw,
        "use_pca": feat_cfg["use_pca"],
        "selector_on_log": feat_cfg["selector_on_log"],
        "var_quantile": feat_cfg["var_quantile"],
        "test_cancer_fn": tm["cancer_fn"],
        "test_cancer_fnr": tm["cancer_fnr"],
        "test_cancer_recall": tm["cancer_recall_sensitivity"],
        "test_f1_macro": tm["f1_macro"],
        "test_accuracy": tm["accuracy"],
        "mlflow_run_id": out["mlflow_run_id"],
    })

res_df = (
    pd.DataFrame(results)
      .sort_values(["test_cancer_fn", "test_cancer_fnr", "test_cancer_recall", "test_f1_macro"], ascending=[True, True, False, False])
      .reset_index(drop=True)
)

res_df.head(15)

2026/01/10 18:40:32 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/10 18:40:32 INFO mlflow.store.db.utils: Updating database tables
2026/01/10 18:40:32 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/10 18:40:32 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/10 18:40:32 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/10 18:40:32 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/10 18:42:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/10 18:45:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/10 19:02:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/10 19:19:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


,model_name,clf_name,mw,use_pca,selector_on_log,var_quantile,test_cancer_fn,test_cancer_fnr,test_cancer_recall,test_f1_macro,test_accuracy,mlflow_run_id
0,logreg_pca0_log0_vq20_mw1p0_sweep,logreg,1.0,False,False,0.2,27,0.082822,0.917178,0.336481,0.518047,68818ddb1fe243d58b82d94fc5957a24
1,logreg_pca0_log0_vq20_mw2p0_sweep,logreg,2.0,False,False,0.2,29,0.088957,0.911043,0.313115,0.513800,6b5802595fd540f5808dd326588e29f5
2,linear_svc_calibrated_pca0_log0_vq20_mw1p0_sweep,linear_svc_calibrated,1.0,False,False,0.2,40,0.122699,0.877301,0.082337,0.416136,c7096af520e54c0e9f79d967312c7d67
3,linear_svc_calibrated_pca0_log0_vq20_mw2p0_sweep,linear_svc_calibrated,2.0,False,False,0.2,41,0.125767,0.874233,0.074617,0.409766,d6fabc6642924b8a83ef643ce0a1ae84


## Entrenamiento final (guardar bundle en `models/`)

In [9]:
# Elegimos el mejor del sweep
best = res_df.iloc[0].to_dict()
best

{'model_name': 'logreg_pca0_log0_vq20_mw1p0_sweep',
 'clf_name': 'logreg',
 'mw': 1.0,
 'use_pca': False,
 'selector_on_log': False,
 'var_quantile': 0.2,
 'test_cancer_fn': 27,
 'test_cancer_fnr': 0.08282208588957055,
 'test_cancer_recall': 0.9171779141104295,
 'test_f1_macro': 0.3364810738704107,
 'test_accuracy': 0.5180467091295117,
 'mlflow_run_id': '68818ddb1fe243d58b82d94fc5957a24'}

In [10]:
best_cfg = replace(
    base_cfg,
    model_name="multiclass_final",
    model_version="v0.1.0",
    clf_name=best["clf_name"],
    malignant_weight=float(best["mw"]),
    use_pca=bool(best["use_pca"]),
    selector_on_log=bool(best["selector_on_log"]),
    var_quantile=float(best["var_quantile"]),
    save_local_bundle=True,
    save_plots=True,
)

final_out = run_training(best_cfg, feature_cols=gene_cols)
final_out

2026/01/10 19:33:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


{'mlflow_run_id': '1b8018f5eb6f4729b3c1b53fab847ef6',
 'cv_metrics': {'accuracy': 0.5335106382978724,
  'balanced_accuracy': 0.3569369900055361,
  'f1_macro': 0.36539868540041875,
  'f1_weighted': 0.521821950441559,
  'log_loss': 1.7644911981627975,
  'cancer_threshold': 0.051448588199762835,
  'cancer_tn': 305,
  'cancer_fp': 273,
  'cancer_fn': 63,
  'cancer_tp': 1239,
  'cancer_fnr': 0.04838709677419355,
  'cancer_recall_sensitivity': 0.9516129032258065,
  'cancer_specificity': 0.527681660899654,
  'cancer_precision': 0.8194444444444444,
  'cancer_roc_auc': 0.8889384976001785,
  'cancer_pr_auc': 0.9443006740643846,
  'per_class_report': {'Breast cancer': {'precision': 0.3492063492063492,
    'recall': 0.55,
    'f1-score': 0.42718446601941745,
    'support': 80.0},
   'Cholangiocarcinoma': {'precision': 0.24528301886792453,
    'recall': 0.18309859154929578,
    'f1-score': 0.20967741935483872,
    'support': 71.0},
   'Colorectal cancer': {'precision': 0.4117647058823529,
    'reca